# Clase 3 — De los datos a un servicio de inferencia

Caso practico: estimar el **precio de un vehiculo usado** (regresion).

Enfoque: como preparar datos, entrenar un baseline, evaluarlo con metricas
interpretables y dejarlo listo para un servicio de inferencia (API).

## 1) ¿Regresion o clasificacion?

La primera decision no es que biblioteca usar, sino **que queremos predecir**.

| Pregunta | Tipo | Ejemplo |
|---|---|---|
| ¿Cuanto costara? | Regresion | Precio de un automovil |
| ¿Ocurrira? | Clasificacion | Contratara: Si/No |
| ¿A que grupo pertenece? | Clasificacion | Tipo de cliente |

En esta clase seguimos un problema completo de **regresion** (precio de
vehiculos) de principio a fin, y al final contrastamos brevemente Regresion
Logistica y KNN para ver como cambia el enfoque cuando el target ya no es
numerico.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', None)

## 2) Recibo un dataset: revision en 5 minutos

Antes de modelar necesitamos responder rapido:

- ¿cuantos datos tengo? (`shape`)
- ¿que significa cada variable? (`dtypes`)
- ¿que esta incompleto? (`isna().sum()`)
- ¿que parece incorrecto? (`duplicated().sum()`, `describe()`)
- ¿cual es mi target? → **`precio`**

Y antes de seguir, verificar unidades: `precio` en MXN, `km` en kilometros.
Una inconsistencia de unidades puede destruir un modelo.

In [ ]:
MODELOS_POR_MARCA = {
    'Nissan': ['Sentra', 'Versa'],
    'Mazda': ['CX-5', 'Mazda 3'],
    'Volkswagen': ['Jetta', 'Vento'],
    'Toyota': ['Corolla', 'Yaris'],
    'Chevrolet': ['Aveo', 'Onix'],
}

PRECIO_BASE_MARCA = {
    'Nissan': 260000,
    'Mazda': 300000,
    'Volkswagen': 270000,
    'Toyota': 290000,
    'Chevrolet': 230000,
}


def build_synthetic_dataset(seed: int = 42, n_rows: int = 1500) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    marcas = list(PRECIO_BASE_MARCA.keys())

    rows = []
    for _ in range(n_rows):
        marca = rng.choice(marcas)
        modelo = rng.choice(MODELOS_POR_MARCA[marca])
        anio = int(rng.integers(2015, 2024))
        km = int(max(1000, rng.normal((2024 - anio) * 12000, 9000)))
        transmision = rng.choice(['Automatica', 'Manual'], p=[0.65, 0.35])

        precio = (
            PRECIO_BASE_MARCA[marca]
            + (anio - 2015) * 18000
            - km * 1.1
            + (5000 if transmision == 'Automatica' else 0)
            # ruido amplio: representa factores no capturados (estado real, mantenimiento)
            + rng.normal(0, 30000)
        )
        precio = float(np.clip(precio, 60000, None))

        rows.append({
            'marca': marca,
            'modelo': modelo,
            'anio': anio,
            'km': km,
            'transmision': transmision,
            'precio': round(precio, 2),
        })

    return pd.DataFrame(rows)


df = build_synthetic_dataset(seed=42, n_rows=1500)
print('shape:', df.shape)
df.head()

In [ ]:
print('dtypes:')
print(df.dtypes)
print()
print('valores nulos:')
print(df.isna().sum())
print()
print('duplicados:', df.duplicated().sum())

In [ ]:
df.describe()

In [ ]:
print('marca:')
print(df['marca'].value_counts())
print()
print('transmision:')
print(df['transmision'].value_counts())

## 3) Estadistica util antes de modelar

No necesitamos calcular todo. Necesitamos detectar comportamiento relevante:

- **Tendencia central**: media vs. mediana de `precio`.
- **Dispersion**: desviacion estandar / IQR de `km`.
- **Posicion**: percentiles.
- **Relacion**: correlacion `anio` ↔ `precio` (esperada positiva) y
  `km` ↔ `precio` (esperada negativa).

Un vehiculo con precio muy alto no se elimina automaticamente: primero
preguntamos si es un error o un vehiculo de alta gama. **Atipico ≠ incorrecto.**

In [ ]:
print('precio -> media:', round(df['precio'].mean(), 2), '| mediana:', round(df['precio'].median(), 2))
print('precio -> std:', round(df['precio'].std(), 2))

q1, q3 = df['km'].quantile([0.25, 0.75])
print('km -> IQR:', round(q3 - q1, 2), '| percentil 90:', round(df['km'].quantile(0.9), 2))

print()
print('correlacion con precio:')
print(df[['anio', 'km', 'precio']].corr()['precio'])

## 4) Preparar los datos que realmente necesita el modelo

Separamos:

- **X** (predictoras): `marca`, `modelo`, `anio`, `km`, `transmision`
- **y** (target): `precio`

Despues: `train` 80% / `test` 20%.

Y por tipo de variable:

- **Categoricas** (`marca`, `modelo`, `transmision`) → encoding (One-Hot).
- **Numericas** (`anio`, `km`) → se usan directo (Regresion Lineal no las
  necesita escaladas, pero mas adelante veremos que **KNN si**).

Dos reglas importantes:

1. El test no debe usarse para aprender transformaciones.
2. La transformacion usada en entrenamiento debe ser exactamente la misma en
   inferencia.

Por eso encapsulamos **preprocessing + modelo** en un `Pipeline`.

In [ ]:
CATEGORICAL_FEATURES = ['marca', 'modelo', 'transmision']
NUMERIC_FEATURES = ['anio', 'km']
FEATURE_ORDER = CATEGORICAL_FEATURES + NUMERIC_FEATURES
TARGET = 'precio'

X = df[FEATURE_ORDER]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ('categorical', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
], remainder='passthrough')

print('train:', X_train.shape, '| test:', X_test.shape)

## 5) Regresion Lineal: nuestro baseline

La Regresion Lineal intenta aproximar una relacion entre las caracteristicas
y una variable numerica (`precio`).

¿Por que empezar aqui?

- sencilla
- rapida
- interpretable
- **util como baseline**

No afirmamos que sea el mejor modelo para valuacion vehicular. Establecemos
una referencia: si mas adelante un modelo mas complejo da un MAE menor,
sabremos cuanto realmente mejora.

In [ ]:
price_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression()),
])

price_model.fit(X_train, y_train)

ejemplo = pd.DataFrame([{
    'marca': 'Nissan',
    'modelo': 'Sentra',
    'anio': 2022,
    'km': 42000,
    'transmision': 'Automatica',
}])
print('Precio estimado (ejemplo):', round(float(price_model.predict(ejemplo)[0]), 2))

## 6) ¿Como se si mi regresion funciona?

- **MAE** (error absoluto medio): facil de comunicar, en las mismas
  unidades que el target (pesos).
- **RMSE**: penaliza mas los errores grandes. Si es mucho mayor que el MAE,
  hay algunos errores especialmente grandes que vale la pena investigar.
- **R²**: que proporcion de la variabilidad del precio explica el modelo.

No preguntar solo "¿R² es alto?". Preguntar: **¿el error es aceptable para
el uso que le quiero dar?** Un MAE de $24,000 puede ser aceptable para un
vehiculo de $1.5M y problematico para uno de $150,000.

In [ ]:
pred = price_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)

print(f'MAE:  ${mae:,.2f} MXN')
print(f'RMSE: ${rmse:,.2f} MXN')
print(f'R2:   {r2:.4f}')

## 7) Cuando el target deja de ser numerico

Contraste rapido con el mismo dataset, ahora prediciendo categorias.

**Regresion Logistica** — ¿el cliente comprara el vehiculo? Target 0/1,
salida `P(compra=1)`. Se evalua con Accuracy, Precision, Recall y F1
(Precision/Recall son especialmente importantes si las clases estan
desbalanceadas).

**KNN** — ¿a que categoria pertenece este vehiculo (Sedan/SUV)? Busca las
`K` observaciones mas cercanas y vota. **KNN depende de distancias**: si
`km` esta en una escala mucho mayor que `anio`, dominara la distancia. Por
eso el escalamiento es especialmente importante para KNN — no existe un
preprocesamiento identico para todos los algoritmos.

In [ ]:
# Regresion Logistica: ¿el cliente comprara el vehiculo?
rng = np.random.default_rng(7)
logit = -1.0 + 0.28 * (df['anio'] - 2015) - 0.00003 * df['km'] + rng.normal(0, 0.6, size=len(df))
prob_compra = 1 / (1 + np.exp(-logit))
df['compra_probable'] = rng.binomial(1, prob_compra)

X_c = df[['anio', 'km', 'transmision']]
y_c = df['compra_probable']
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    X_c, y_c, test_size=0.2, random_state=42, stratify=y_c
)

logit_model = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[
        ('transmision', OneHotEncoder(handle_unknown='ignore'), ['transmision']),
    ], remainder='passthrough')),
    ('classifier', LogisticRegression(max_iter=1000)),
])
logit_model.fit(X_c_train, y_c_train)
pred_c = logit_model.predict(X_c_test)

print('Accuracy: ', round(accuracy_score(y_c_test, pred_c), 4))
print('Precision:', round(precision_score(y_c_test, pred_c, zero_division=0), 4))
print('Recall:   ', round(recall_score(y_c_test, pred_c, zero_division=0), 4))
print('F1:       ', round(f1_score(y_c_test, pred_c, zero_division=0), 4))

In [ ]:
# KNN: ¿es una "buena compra"? (anio reciente y km bajo) — sensible a escala
df['buena_compra'] = ((df['anio'] >= 2020) & (df['km'] <= 60000)).astype(int)
print(df['buena_compra'].value_counts())
print()

X_k = df[['anio', 'km']]
y_k = df['buena_compra']
X_k_train, X_k_test, y_k_train, y_k_test = train_test_split(
    X_k, y_k, test_size=0.2, random_state=42, stratify=y_k
)

knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_k_train, y_k_train)
acc_raw = accuracy_score(y_k_test, knn_raw.predict(X_k_test))

scaler = StandardScaler()
X_k_train_scaled = scaler.fit_transform(X_k_train)
X_k_test_scaled = scaler.transform(X_k_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_k_train_scaled, y_k_train)
acc_scaled = accuracy_score(y_k_test, knn_scaled.predict(X_k_test_scaled))

print(f'KNN sin escalar -> accuracy: {acc_raw:.4f}')
print(f'KNN escalado    -> accuracy: {acc_scaled:.4f}')
print()
print('anio va de 2015 a 2023 (rango ~8). km llega a ~150,000 (rango mucho mayor).')
print('Sin escalar, la distancia la domina km; anio casi no influye en el vecino mas cercano.')

## 8) Del modelo al servicio de inferencia

Esto ocurre **fuera** de la API (entrenamiento offline):

```
Datos historicos → Preparacion → Train/Test → Entrenamiento → Evaluacion → vehicle_price_v1.joblib
```

Y esto ocurre **dentro** de la API (inferencia):

```
Usuario captura marca/modelo/anio/km/transmision
        ↓
POST /predict-price
        ↓
Validacion (Pydantic)
        ↓
Carga pipeline/modelo (joblib)
        ↓
predict()
        ↓
{"estimated_price": 318500, "currency": "MXN", "model_version": "v1"}
        ↓
Frontend
```

**Punto central: entrenar y predecir son procesos diferentes.** La API usa
un modelo ya entrenado; nunca reentrena en cada request.

Guardamos el pipeline que entrenamos arriba para que el servicio de
`proyecto_web_precio_vehiculos/` lo use directamente.

In [ ]:
project_root = Path('proyecto_web_precio_vehiculos')
models_dir = project_root / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(price_model, models_dir / 'vehicle_price_v1.joblib')

metrics = {
    'mae': round(float(mae), 2),
    'rmse': round(float(rmse), 2),
    'r2': round(float(r2), 4),
    'features': FEATURE_ORDER,
    'target': TARGET,
    'model_type': 'LinearRegression',
    'model_version': 'v1',
}
with open(models_dir / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Modelo y metricas guardados en proyecto_web_precio_vehiculos/models')

## 9) Arquitectura minima de una solucion de ML

```
FUENTES (BD, CSV/Parquet, APIs)
        ↓
DATA / TRAINING
  Validacion → Preparacion → Entrenamiento → Evaluacion → Modelo versionado
        ↓
INFERENCE SERVICE (FastAPI /predict-price)
        ↓
Pipeline (Preprocessing + Modelo)
        ↓
CONSUMIDORES (Frontend, otra app, otro sistema)
```

Bloques transversales:

- **Seguridad**: autenticacion (quien consume), validacion (que datos
  aceptamos — ya lo hace Pydantic), secrets fuera del codigo, HTTPS en
  transito.
- **Buenas practicas**: versionar el modelo (`vehicle_price_v1`), logging,
  testing (preprocessing, modelo y endpoint) y monitoring (disponibilidad +
  comportamiento del modelo/datos).

Todo esto ya esta implementado, con el mismo caso de vehiculos, en
`proyecto_web_precio_vehiculos/`. Para correrlo:

```bash
cd proyecto_web_precio_vehiculos
pip install -r requirements.txt
python scripts/train_model.py   # reentrena si hace falta
bash scripts/run_site.sh        # frontend :9010, backend :9011
```